In [0]:
%sql
CREATE OR REPLACE TABLE fooddelivery.gold.kpi_faturamento_mensal AS
SELECT 
    -- Agrupamento por mês e categoria para o dashboard
    date_format(transaction_time, 'yyyy-MM') AS mes_referencia,
    menu_category,
    
    -- 1. Faturamento Bruto (Total absoluto)
    ROUND(SUM(order_price_brl), 2) AS faturamento_bruto_brl,
    
    -- 2. Faturamento Ajustado (Soma apenas onde NÃO é outlier)
    ROUND(SUM(CASE WHEN is_outlier = False THEN order_price_brl ELSE 0 END), 2) AS faturamento_ajustado_brl,
    
    -- 3. Métricas de Qualidade (KPIs de controle)
    COUNT(*) AS total_pedidos,
    COUNT(CASE WHEN is_outlier = True THEN 1 END) AS qtd_outliers,
    
    -- 4. Percentual de Impacto dos Outliers no Faturamento
    ROUND(
        (SUM(CASE WHEN is_outlier = True THEN order_price_brl ELSE 0 END) / SUM(order_price_brl)) * 100, 
        2
    ) AS pct_impacto_outlier

FROM fooddelivery.silver.orders
GROUP BY 1, 2
ORDER BY 1 DESC, faturamento_ajustado_brl DESC;